In [25]:
# Imports
import os
os.environ["JAVA_TOOL_OPTIONS"] = "-Xmx8G"

from datetime import datetime, timedelta
from pathlib import Path
import json
import shutil
import zipfile

import geopandas as gpd
import numpy as np
import pandas as pd
import requests
import r5py
from shapely.geometry import box

In [26]:
# Paths
GTFS_URL = "https://data.opentransportdata.swiss/fr/dataset/timetable-2026-gtfs2020/permalink"
GTFS_DIR = Path("data/gtfs")
ZIP_FILE = GTFS_DIR / "gtfs_fp2026.zip"
FILTERED_ZIP_FILE = GTFS_DIR / "gtfs_filtered_no_taxi.zip"

OUTPUTS_DIR = Path("../outputs")
PUBLIC_DATA_DIR = Path("public/data")
PROCESSED_DATA_DIR = Path("data/processed")

for directory in [GTFS_DIR, OUTPUTS_DIR, PUBLIC_DATA_DIR, PROCESSED_DATA_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

In [27]:
# Download GTFS data when needed
if not ZIP_FILE.exists():
    print(f"Downloading GTFS data to {ZIP_FILE}...")
    response = requests.get(GTFS_URL, stream=True)
    response.raise_for_status()

    with open(ZIP_FILE, "wb") as file:
        for chunk in response.iter_content(chunk_size=8192):
            file.write(chunk)
else:
    print(f"GTFS file already exists at {ZIP_FILE}")

GTFS file already exists at data/gtfs/gtfs_fp2026.zip


In [28]:
# Extract GTFS files
with zipfile.ZipFile(ZIP_FILE, "r") as zip_ref:
    zip_ref.extractall(GTFS_DIR)

print(f"GTFS files extracted to {GTFS_DIR}")

GTFS files extracted to data/gtfs


In [29]:
# Download OSM data when needed
OSM_URL = "https://download.geofabrik.de/europe/switzerland-latest.osm.pbf"
OSM_FILE = Path("data/switzerland-latest.osm.pbf")

if not OSM_FILE.exists():
    print(f"Downloading OSM data to {OSM_FILE}...")
    response = requests.get(OSM_URL, stream=True)
    response.raise_for_status()

    total_size = int(response.headers.get("content-length", 0))
    downloaded = 0

    with open(OSM_FILE, "wb") as file:
        for chunk in response.iter_content(chunk_size=1024 * 1024):
            file.write(chunk)
            downloaded += len(chunk)
            print(
                f"\rProgress: {downloaded / (1024 * 1024):.1f}/{total_size / (1024 * 1024):.1f} MB",
                end="",
            )

    print(f"\nOSM data downloaded to {OSM_FILE}")
else:
    print(f"OSM file already exists at {OSM_FILE}")

OSM file already exists at data/switzerland-latest.osm.pbf


In [30]:
SWISS_CRS = "EPSG:2056"   # LV95, meters
WGS84_CRS = "EPSG:4326"   # lon/lat
GRID_SIZE_M = 2000        # 2000 m cells
MAX_TIME_MINUTES = [30, 60, 90, 120, 150, 180, 210, 240]
DEPARTURE_TIMES = [9, 12, 17, 22]
FILTER_ORIGINS_BY_TRANSIT_PROXIMITY = True
MAX_ORIGIN_DISTANCE_TO_TRANSIT_M = 5000
RUN_DATE = datetime(2026, 3, 20)

In [31]:
# Load and dissolve municipal boundaries into one Switzerland boundary
municipality_boundaries = gpd.read_file(
    "../data/Boundaries_K4_Commune_20260101.gpkg",
    layer="boundaries",
).to_crs(SWISS_CRS)

switzerland_boundary = municipality_boundaries.dissolve().reset_index(drop=True)

In [32]:
# Load GTFS tables
stops = pd.read_csv(GTFS_DIR / "stops.txt")
routes = pd.read_csv(GTFS_DIR / "routes.txt")
trips = pd.read_csv(GTFS_DIR / "trips.txt")
stop_times = pd.read_csv(GTFS_DIR / "stop_times.txt")
transfers = pd.read_csv(GTFS_DIR / "transfers.txt", dtype=str)

# Remove taxi routes and orphaned related records
routes = routes[routes["route_type"] != 1500].copy()
trips = trips[trips["route_id"].isin(routes["route_id"])].copy()
stop_times = stop_times[stop_times["trip_id"].isin(trips["trip_id"])].copy()
stops = stops[stops["stop_id"].isin(stop_times["stop_id"])].copy()

valid_stop_ids = set(stops["stop_id"].astype(str))
transfers = transfers[
    transfers["from_stop_id"].isin(valid_stop_ids)
    & transfers["to_stop_id"].isin(valid_stop_ids)
].copy()

print("GTFS tables loaded and filtered.")

GTFS tables loaded and filtered.


In [33]:
# Save filtered GTFS feed for r5py network creation
with zipfile.ZipFile(FILTERED_ZIP_FILE, "w", compression=zipfile.ZIP_DEFLATED) as zip_file:
    zip_file.writestr("routes.txt", routes.to_csv(index=False))
    zip_file.writestr("trips.txt", trips.to_csv(index=False))
    zip_file.writestr("stop_times.txt", stop_times.to_csv(index=False))
    zip_file.writestr("stops.txt", stops.to_csv(index=False))
    zip_file.writestr("transfers.txt", transfers.to_csv(index=False))

    optional_gtfs_files = [
        "agency.txt",
        "calendar.txt",
        "calendar_dates.txt",
        "feed_info.txt",
        "pathways.txt",
        "levels.txt",
        "shapes.txt",
        "frequencies.txt",
        "attributions.txt",
    ]

    for filename in optional_gtfs_files:
        source_file = GTFS_DIR / filename
        if source_file.exists():
            zip_file.write(source_file, arcname=filename)

In [34]:
def build_swiss_grid(boundary_gdf: gpd.GeoDataFrame, cell_size: int = 500):
    xmin, ymin, xmax, ymax = boundary_gdf.total_bounds
    xs = np.arange(xmin, xmax, cell_size)
    ys = np.arange(ymin, ymax, cell_size)

    cells = []
    ids = []

    for cell_id, (x, y) in enumerate((x, y) for x in xs for y in ys):
        cells.append(box(x, y, x + cell_size, y + cell_size))
        ids.append(str(cell_id))

    grid = gpd.GeoDataFrame(
        {"id": ids},
        geometry=cells,
        crs=boundary_gdf.crs,
    )

    grid = gpd.overlay(grid, boundary_gdf, how="intersection")

    origins = grid.copy()
    origins["geometry"] = origins.geometry.centroid

    return grid.reset_index(drop=True), origins.reset_index(drop=True)

In [35]:
grid_cells_3000m_lv95, routing_points_lv95 = build_swiss_grid(
    switzerland_boundary,
    cell_size=GRID_SIZE_M,
)

grid_cells_3000m_lv95["geometry"] = grid_cells_3000m_lv95.geometry.make_valid()
grid_cells_3000m_lv95 = grid_cells_3000m_lv95[
    grid_cells_3000m_lv95.geometry.geom_type.isin(["Polygon", "MultiPolygon"])
].copy()
grid_cells_3000m_lv95 = grid_cells_3000m_lv95[
    ~grid_cells_3000m_lv95.geometry.is_empty
].copy()

origin_filter_metadata = {
    "enabled": FILTER_ORIGINS_BY_TRANSIT_PROXIMITY,
    "max_distance_to_transit_m": MAX_ORIGIN_DISTANCE_TO_TRANSIT_M,
    "n_origins_before_filter": int(len(routing_points_lv95)),
}

if FILTER_ORIGINS_BY_TRANSIT_PROXIMITY:
    transit_stops = stops.dropna(subset=["stop_lon", "stop_lat"]).copy()
    transit_stops_gdf = gpd.GeoDataFrame(
        {"stop_id": transit_stops["stop_id"].astype(str)},
        geometry=gpd.points_from_xy(
            transit_stops["stop_lon"].astype(float),
            transit_stops["stop_lat"].astype(float),
        ),
        crs=WGS84_CRS,
    ).to_crs(SWISS_CRS)

    nearest_stop = gpd.sjoin_nearest(
        routing_points_lv95[["id", "geometry"]],
        transit_stops_gdf,
        how="left",
        max_distance=MAX_ORIGIN_DISTANCE_TO_TRANSIT_M,
        distance_col="nearest_transit_stop_m",
    )
    nearest_stop_distance = (
        nearest_stop.dropna(subset=["stop_id"])
        .groupby("id", as_index=False)["nearest_transit_stop_m"]
        .min()
    )

    keep_origin_ids = set(nearest_stop_distance["id"].astype(str))
    grid_cells_3000m_lv95 = grid_cells_3000m_lv95[
        grid_cells_3000m_lv95["id"].astype(str).isin(keep_origin_ids)
    ].copy()
    routing_points_lv95 = routing_points_lv95[
        routing_points_lv95["id"].astype(str).isin(keep_origin_ids)
    ].copy()
    routing_points_lv95 = routing_points_lv95.merge(
        nearest_stop_distance,
        on="id",
        how="left",
    )

origin_filter_metadata["n_origins_after_filter"] = int(len(routing_points_lv95))
origin_filter_metadata["n_origins_removed"] = (
    origin_filter_metadata["n_origins_before_filter"]
    - origin_filter_metadata["n_origins_after_filter"]
)

routing_points = routing_points_lv95.to_crs(WGS84_CRS)

print("Number of 3000m cells:", len(grid_cells_3000m_lv95))
print("Number of routing origins:", len(routing_points))
print("Origins removed by transit proximity filter:", origin_filter_metadata["n_origins_removed"])

Number of 3000m cells: 10203
Number of routing origins: 10203
Origins removed by transit proximity filter: 559


In [36]:
# Select InterCity routes and the stops where they serve passengers
ic_routes = routes[routes["route_desc"] == "IC"].copy()
ic_trips = trips[trips["route_id"].isin(ic_routes["route_id"])].copy()
ic_stop_ids = stop_times[stop_times["trip_id"].isin(ic_trips["trip_id"])]["stop_id"].unique()

In [37]:
# Convert IC platform stops to station-level destination points
ic_stops = stops[stops["stop_id"].isin(ic_stop_ids.astype(str))].copy()
ic_stops["station_id"] = ic_stops["parent_station"].where(
    ic_stops["parent_station"].notna() & (ic_stops["parent_station"] != ""),
    ic_stops["stop_id"],
).astype(str)
ic_stops["station_id"] = ic_stops["station_id"].str.replace(r"^Parent", "", regex=True)

ic_station_ids = ic_stops["station_id"].dropna().unique()
ic_stations = stops[stops["stop_id"].astype(str).isin(ic_station_ids)].copy()
ic_stations = ic_stations.drop_duplicates(subset=["stop_id"]).copy()

ic_hubs = gpd.GeoDataFrame(
    {
        "id": ic_stations["stop_id"].astype(str),
        "stop_id": ic_stations["stop_id"].astype(str),
        "stop_name": ic_stations["stop_name"],
    },
    geometry=gpd.points_from_xy(
        ic_stations["stop_lon"].astype(float),
        ic_stations["stop_lat"].astype(float),
    ),
    crs=WGS84_CRS,
).reset_index(drop=True)

print("IC destinations:", len(ic_hubs))

IC destinations: 139


In [38]:
# Check Java availability for r5py
java_check = os.popen("java -version 2>&1").read()

if "version" in java_check.lower():
    print(f"Java detected: {java_check.splitlines()[0]}")
    print("r5py is ready.")
else:
    raise RuntimeError("Java was not detected. Install OpenJDK before running r5py.")

Java detected: Picked up JAVA_TOOL_OPTIONS: -Xmx8G
r5py is ready.


In [39]:
# Build the multimodal transport network
network = r5py.TransportNetwork(
    str(OSM_FILE),
    [str(FILTERED_ZIP_FILE)],
)

In [40]:
def time_label(hour: int) -> str:
    return f"{hour:02d}00"

def max_time_label(minutes: int) -> str:
    return f"{minutes:03d}min"

grid_label = f"{GRID_SIZE_M}m"
MAX_TIME_FULL = max(MAX_TIME_MINUTES)  # 240 min — compute once per departure

travel_time_outputs = []
travel_times_by_combo = {}

for departure_hour in DEPARTURE_TIMES:
    departure_label = time_label(departure_hour)
    full_max_label = max_time_label(MAX_TIME_FULL)

    # Cache file for the full (240 min) run
    full_out_file = OUTPUTS_DIR / f"travel_times_{grid_label}_departure_{departure_label}_max_{full_max_label}.parquet"

    if full_out_file.exists():
        print(f"Loading full matrix for {departure_label}: {full_out_file}")
        full_matrix = pd.read_parquet(full_out_file)
    else:
        print(f"Computing full matrix for {departure_label}, max {MAX_TIME_FULL} min...")
        full_matrix = r5py.TravelTimeMatrix(
            network,
            origins=routing_points,
            destinations=ic_hubs,
            departure=datetime(RUN_DATE.year, RUN_DATE.month, RUN_DATE.day, departure_hour, 0),
            departure_time_window=timedelta(minutes=60),
            transport_modes=[r5py.TransportMode.WALK, r5py.TransportMode.TRANSIT],
            percentiles=[50],
            snap_to_network=True,
            max_time=timedelta(minutes=MAX_TIME_FULL),
        )
        full_matrix = full_matrix.copy()
        full_matrix["departure_label"] = departure_label
        full_matrix["departure_hour"] = departure_hour
        full_matrix.to_parquet(full_out_file, index=False)
        print(f"Saved full matrix to {full_out_file}")

    # Derive all max_time thresholds by filtering the full matrix in pandas
    for max_time_minutes in MAX_TIME_MINUTES:
        max_label = max_time_label(max_time_minutes)
        out_file = OUTPUTS_DIR / f"travel_times_{grid_label}_departure_{departure_label}_max_{max_label}.parquet"

        if out_file.exists():
            print(f"Loading saved: {out_file}")
            matrix = pd.read_parquet(out_file)
        else:
            # Filter to only rows within this time threshold
            matrix = full_matrix[
                full_matrix["travel_time"].isna() | (full_matrix["travel_time"] <= max_time_minutes)
            ].copy()
            matrix["max_time_minutes"] = max_time_minutes
            matrix.to_parquet(out_file, index=False)
            print(f"Derived and saved: {out_file}")

        travel_times_by_combo[(departure_label, max_time_minutes)] = matrix
        travel_time_outputs.append({
            "departure_label": departure_label,
            "departure_hour": departure_hour,
            "max_time_minutes": max_time_minutes,
            "file": str(out_file),
            "n_rows": int(len(matrix)),
        })

travel_times = pd.concat(travel_times_by_combo.values(), ignore_index=True)
print("Saved", len(travel_time_outputs), "travel-time matrices")

Loading full matrix for 0900: ../outputs/travel_times_2000m_departure_0900_max_240min.parquet
Loading saved: ../outputs/travel_times_2000m_departure_0900_max_030min.parquet
Loading saved: ../outputs/travel_times_2000m_departure_0900_max_060min.parquet
Loading saved: ../outputs/travel_times_2000m_departure_0900_max_090min.parquet
Loading saved: ../outputs/travel_times_2000m_departure_0900_max_120min.parquet
Loading saved: ../outputs/travel_times_2000m_departure_0900_max_150min.parquet
Loading saved: ../outputs/travel_times_2000m_departure_0900_max_180min.parquet
Loading saved: ../outputs/travel_times_2000m_departure_0900_max_210min.parquet
Loading saved: ../outputs/travel_times_2000m_departure_0900_max_240min.parquet
Loading full matrix for 1200: ../outputs/travel_times_2000m_departure_1200_max_240min.parquet
Loading saved: ../outputs/travel_times_2000m_departure_1200_max_030min.parquet
Loading saved: ../outputs/travel_times_2000m_departure_1200_max_060min.parquet
Loading saved: ../outp

In [ ]:
# Save inputs, OD pairs, accessibility summaries, and run metadata
grid_cells_file = OUTPUTS_DIR / f"grid_cells_{grid_label}_lv95.parquet"
grid_points_file = OUTPUTS_DIR / f"grid_points_{grid_label}.parquet"

grid_cells_3000m_lv95.to_parquet(grid_cells_file, index=False)
routing_points.to_parquet(grid_points_file, index=False)
ic_hubs.to_parquet(OUTPUTS_DIR / "ic_stations.parquet", index=False)

shutil.copy(FILTERED_ZIP_FILE, PROCESSED_DATA_DIR / "gtfs_filtered_no_taxi.zip")

# --- Attach municipality attributes to each origin point ---
muni_cols_to_drop = ["GDENAME", "KTNAME", "KTKZ", "GDEHISTID", "GDENR", "KTNR"]
routing_points_clean = routing_points.drop(
    columns=[c for c in muni_cols_to_drop if c in routing_points.columns],
    errors="ignore",
)

routing_points_with_muni = gpd.sjoin(
    routing_points_clean.to_crs(SWISS_CRS),
    municipality_boundaries[["GDENAME", "KTNAME", "KTKZ", "geometry"]],
    how="left",
    predicate="within",
).drop(columns=["index_right"], errors="ignore").drop_duplicates(subset=["id"]).to_crs(WGS84_CRS)


valid_times_by_combo = {}
accessibility_summaries_by_combo = {}
summary_outputs = []

for (departure_label, max_time_minutes), combo_times in travel_times_by_combo.items():
    max_label = max_time_label(max_time_minutes)
    combo_stem = f"{grid_label}_departure_{departure_label}_max_{max_label}"

    valid_times = combo_times.dropna(subset=["travel_time"]).copy()
    valid_times["from_id"] = valid_times["from_id"].astype(str)
    valid_times["to_id"] = valid_times["to_id"].astype(str)
    valid_times = valid_times[valid_times["travel_time"] <= max_time_minutes].copy()

    od_file = OUTPUTS_DIR / f"od_pairs_{combo_stem}.parquet"
    valid_times.to_parquet(od_file, index=False)
    valid_times_by_combo[(departure_label, max_time_minutes)] = valid_times

    count_col = f"n_destinations_reachable_{max_label}"
    summary = (
        valid_times.groupby("from_id", as_index=False)
        .agg(
            min_travel_time_to_ic=("travel_time", "min"),
            **{count_col: ("to_id", "nunique")},
        )
    )

    accessibility_summary = routing_points_with_muni.copy()
    accessibility_summary["id"] = accessibility_summary["id"].astype(str)
    accessibility_summary = accessibility_summary.merge(
        summary,
        left_on="id",
        right_on="from_id",
        how="left",
    )
    accessibility_summary[count_col] = accessibility_summary[count_col].fillna(0).astype(int)

    summary_file = OUTPUTS_DIR / f"accessibility_summary_{combo_stem}.parquet"
    accessibility_summary.to_parquet(summary_file, index=False)
    accessibility_summaries_by_combo[(departure_label, max_time_minutes)] = accessibility_summary

    summary_outputs.append(
        {
            "departure_label": departure_label,
            "max_time_minutes": max_time_minutes,
            "od_pairs_file": str(od_file),
            "accessibility_summary_file": str(summary_file),
            "n_valid_rows": int(len(valid_times)),
            "n_overlay_points": int(len(accessibility_summary)),
        }
    )

ic_hubs.to_file(PUBLIC_DATA_DIR / "ic_hubs.geojson", driver="GeoJSON")

metadata = {
    "run_date": RUN_DATE.strftime("%Y-%m-%d"),
    "departure_times": [f"{hour:02d}:00" for hour in DEPARTURE_TIMES],
    "departure_time_window_minutes": 60,
    "max_time_minutes": MAX_TIME_MINUTES,
    "transport_modes": ["WALK", "TRANSIT"],
    "percentiles": [50],
    "grid_resolution_meters": GRID_SIZE_M,
    "grid_crs": SWISS_CRS,
    "routing_crs": WGS84_CRS,
    "osm_file": str(OSM_FILE) if "OSM_FILE" in globals() else None,
    "gtfs_file": "data/processed/gtfs_filtered_no_taxi.zip",
    "n_origins": int(len(routing_points)),
    "n_destinations": int(len(ic_hubs)),
    "origin_filter": origin_filter_metadata,
    "travel_time_outputs": travel_time_outputs,
    "summary_outputs": summary_outputs,
}

metadata_file = OUTPUTS_DIR / f"run_metadata_{grid_label}_time_sweep.json"
with open(metadata_file, "w", encoding="utf-8") as file:
    json.dump(metadata, file, indent=2, ensure_ascii=False)

print("Saved:")
print("-", grid_cells_file)
print("-", grid_points_file)
print("-", OUTPUTS_DIR / "ic_stations.parquet")
print("-", len(travel_time_outputs), "travel-time matrix files")
print("-", len(summary_outputs), "OD-pair and accessibility summary files")
print("-", metadata_file)


df = pd.read_parquet(OUTPUTS_DIR / f"accessibility_summary_{grid_label}_departure_0900_max_030min.parquet")
print("\nColumns:", df.columns.tolist())
print("\nSample GDENAME values:", df["GDENAME"].head(10).tolist())

Saved:
- ../outputs/grid_cells_2000m_lv95.parquet
- ../outputs/grid_points_2000m.parquet
- ../outputs/ic_stations.parquet
- 32 travel-time matrix files
- 32 OD-pair and accessibility summary files
- ../outputs/run_metadata_2000m_time_sweep.json

Columns: ['id', 'geometry', 'nearest_transit_stop_m', 'GDENAME', 'KTNAME', 'KTKZ', 'from_id', 'min_travel_time_to_ic', 'n_destinations_reachable_030min']

Sample GDENAME values: ['Chancy', 'Chancy', 'Avully', 'Dardagny', 'Dardagny', 'Dardagny', nan, 'Avusy', 'Avully', 'Dardagny']
